# Page through Q3

*Data Analysis in Python · Topic 12.2*

Your media team wants Q3 2025's daily numbers by channel. Page through the report until the last page, and check that you got every row.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell. It reads the course API key from Colab Secrets, as in topic 12.1.
2. In the first cell marked ✏️, page through the `reports/daily` endpoint for `START_DATE` to `END_DATE`, with `channel` as the dimension and `PAGE_SIZE` rows per page. Collect every page's rows in `rows`, and count the pages in `pages`.
3. In the second cell marked ✏️, create the `q3` table from `rows`, and `row_count`: the `row_count` the API reported.
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

If a request stops with `429`, you've sent more than 10 requests in 10 seconds. If it stops with `503`, the API was briefly unavailable, which happens to about 3% of requests. Either way, wait 10 seconds and run the cell again. The next topic handles both for you.

**Expected output**: `pages` is `9`. The `q3` table has 828 rows, the same as `row_count`, with 333,079 sessions and $712,340.64 of revenue.

## Parameters

In [ ]:
import pandas as pd
import requests
from google.colab import userdata

BASE_URL = "https://api.teamsimmer.com/v1/"
API_KEY = userdata.get("SIMMER_API_KEY")  # Colab Secrets, as in topic 12.1
HEADERS = {"X-API-Key": API_KEY}
START_DATE, END_DATE = "2025-07-01", "2025-09-30"
PAGE_SIZE = 100

## Every page

In [ ]:
params = {"start_date": START_DATE, "end_date": END_DATE, "dimensions": "channel", "page_size": PAGE_SIZE}
rows = []
pages = 0
while True:
    response = requests.get(BASE_URL + "reports/daily", headers=HEADERS, params=params)
    response.raise_for_status()
    body = response.json()
    rows.extend(body["rows"])
    pages += 1
    if "next_page_token" not in body:
        break
    params["page_token"] = body["next_page_token"]
pages, len(rows)

## The table

In [ ]:
q3 = pd.DataFrame(rows)
row_count = body["row_count"]
len(q3) == row_count, q3["sessions"].sum()

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    rows, pages = found.get("rows"), found.get("pages")
    if not isinstance(rows, list) or not isinstance(pages, int):
        results.append((False, "rows should be a list of every page's rows, and pages the number of pages you asked for."))
    elif pages == 1 or len(rows) == 100:
        results.append((False, "You have only the first page: 100 rows. Keep asking, with each answer's next_page_token as page_token, until an answer has no token."))
    elif rows and isinstance(rows[0], list):
        results.append((False, "rows is a list of pages, not a list of rows. Use rows.extend(body[\"rows\"]) instead of append()."))
    elif pages != 9 or len(rows) != 828:
        results.append((False, f"Paging through Q3 by channel, 100 rows at a time, takes 9 pages and gives 828 rows. You have {pages} pages and {len(rows)} rows."))
    else:
        results.append((True, "9 pages, 828 rows."))

    q3, row_count = found.get("q3"), found.get("row_count")
    if not isinstance(q3, pd.DataFrame) or "sessions" not in q3.columns:
        results.append((False, "q3 should be a DataFrame: pd.DataFrame(rows)."))
    elif row_count != 828:
        results.append((False, f"row_count should be the row_count the API reported: body[\"row_count\"], 828. It is {row_count!r}."))
    elif len(q3) != row_count:
        results.append((False, f"The q3 table has {len(q3)} rows, and the API reported {row_count}. Page through every page."))
    elif int(q3["sessions"].sum()) != 333079 or abs(q3["revenue"].sum() - 712340.64) > 0.01:
        results.append((False, f"The q3 table should have 333,079 sessions and $712,340.64 of revenue, from July 1 to September 30, 2025. It has {q3['sessions'].sum():,} and ${q3['revenue'].sum():,.2f}."))
    else:
        results.append((True, "828 rows, as the API reported: 333,079 sessions and $712,340.64 of revenue."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
params = {"start_date": START_DATE, "end_date": END_DATE, "dimensions": "channel", "page_size": PAGE_SIZE}
rows = []
pages = 0
while True:
    response = requests.get(BASE_URL + "reports/daily", headers=HEADERS, params=params)
    response.raise_for_status()
    body = response.json()
    rows.extend(body["rows"])
    pages += 1
    if "next_page_token" not in body:
        break
    params["page_token"] = body["next_page_token"]
pages, len(rows)
```

```python
q3 = pd.DataFrame(rows)
row_count = body["row_count"]
len(q3) == row_count, q3["sessions"].sum()
```

</details>